# Aula 04 — Diagnósticos estatísticos e aderência à NBR 14653 (Versão professor)

Notebook de condução docente para a quarta aula do treinamento inferencial.

## Finalidade desta versão

Esta edição foi pensada para o professor e, por isso, contém:

- explicações pedagógicas sobre validação do modelo OLS;
- notas de condução oral sobre significância, normalidade e independência;
- alertas contra leitura mecânica de testes estatísticos;
- perguntas para discussão em sala;
- fechamento do ciclo entre coleta, sanitização, ajuste e validação.

## Resultado esperado da aula

Ao final da condução, a turma deve compreender que um modelo ajustado só ganha sustentação técnica quando seus diagnósticos estatísticos e critérios de aderência são avaliados de forma explícita e documentada.

## Roteiro sugerido de tempo

- **0 a 6 min** — retomada do modelo ajustado na Aula 3;
- **6 a 15 min** — apresentação dos critérios de validação;
- **15 a 28 min** — leitura do relatório de diagnósticos NBR;
- **28 a 38 min** — leitura da significância individual dos coeficientes;
- **38 a 46 min** — discussão sobre normalidade e independência dos resíduos;
- **46 a 50 min** — fechamento técnico e síntese do ciclo completo.

## Estratégia didática

Nesta aula, o professor deve sustentar quatro mensagens centrais:

1. ajustar modelo não basta; é preciso validá-lo;
2. testes estatísticos orientam decisão, mas não dispensam interpretação;
3. aprovação global e significância individual são leituras complementares;
4. o relatório final deve ser comunicável tecnicamente.

In [1]:
# Importações da aula.
# Professor: esta é a culminância do fluxo didático.
# As quatro aulas agora se encadeiam claramente: coleta,
# unitarização, sanitização, regressão e validação.

from __future__ import annotations

from pathlib import Path
from typing import Final

import pandas as pd

from servicos.carregamento import load_raw_dataset, resolve_project_root
from servicos.unitarizacao import add_unit_price_column
from servicos.regressao import fit_ols_regression
from servicos.diagnosticos import (
    build_nbr_diagnostics_report,
    format_coefficients_for_display,
    format_diagnostics_for_display,
)

## Nota de condução oral

Sugestão de fala:

“Na Aula 3 nós estimamos uma equação. Hoje vamos responder se essa equação se sustenta tecnicamente. É aqui que a modelagem deixa de ser apenas ajuste numérico e passa a ser objeto de validação.”

In [2]:
# Mesma estratégia de localização da base das aulas anteriores.
# Isso reforça continuidade e reduz ruído operacional.

DATASET_CANDIDATES: Final[tuple[str, ...]] = (
    'amostras_residencial35.csv',
    'amostrasresidencial35.csv',
    'amostras_residencial.csv',
)

TARGET_COLUMN: Final[str] = 'preco'
PREFERRED_FEATURES: Final[tuple[str, ...]] = (
    'areaprivativa',
    'vagas',
    'distanciacentrokm',
    'dist_praia',
)


def locate_default_dataset(project_root: Path) -> Path:
    """Localiza automaticamente a base padrão das aulas iniciais."""
    data_dir = project_root / 'data'

    for filename in DATASET_CANDIDATES:
        candidate = data_dir / filename
        if candidate.exists():
            return candidate

    searched = ', '.join(DATASET_CANDIDATES)
    raise FileNotFoundError(
        'Nenhum arquivo padrão foi encontrado na pasta `data`. '
        f'Arquivos procurados: {searched}.'
    )

## Etapa 1 — Resolver a base e preparar a amostra

### Intenção docente

Mesmo na aula de diagnósticos, é útil reforçar de onde o modelo nasce.
Isso ajuda a turma a enxergar a validação como continuação do processo, e não como um apêndice isolado.

In [3]:
project_root = resolve_project_root()
dataset_path = locate_default_dataset(project_root)

df_raw = load_raw_dataset(dataset_path)
df_prepared = add_unit_price_column(df_raw)

print('Raiz do projeto:', project_root)
print('Arquivo utilizado:', dataset_path)
print('Dimensão da base preparada:', df_prepared.shape)

Raiz do projeto: /Users/elydocarmobarros/Desktop/ESTUDOS TEC/JUPYTER PROJECTS/treinamento_inferencia
Arquivo utilizado: /Users/elydocarmobarros/Desktop/ESTUDOS TEC/JUPYTER PROJECTS/treinamento_inferencia/data/amostras_residencial.csv
Dimensão da base preparada: (20, 8)


## Etapa 2 — Ajustar novamente o modelo-base da Aula 3

### Por que fazer isso aqui

A Aula 4 depende de um modelo ajustado para poder avaliá-lo.
Por isso, a reconstrução do modelo dentro do notebook é parte da lógica pedagógica e da reprodutibilidade técnica.

In [4]:
available_features = [column for column in PREFERRED_FEATURES if column in df_prepared.columns]

artifacts = fit_ols_regression(
    df=df_prepared,
    target_col=TARGET_COLUMN,
    feature_columns=available_features,
    add_intercept=True,
)

artifacts

OLSRegressionArtifacts(coefficients=const               -549422.227260
areaprivativa         16399.584411
vagas                 17567.447399
distanciacentrokm    -40893.354601
Name: coeficiente, dtype: float64, fitted_values=0     8.306053e+05
1     9.290662e+05
2     6.450794e+05
3     1.356518e+06
4     3.623661e+05
5     1.146329e+06
6     6.942359e+05
7     5.262352e+05
8     1.450784e+06
9     1.015175e+06
10    4.607214e+05
11    8.716254e+05
12    1.080688e+06
13    5.756452e+05
14    9.618442e+05
15    8.142902e+05
16    6.204589e+05
17    1.237631e+06
18    4.401902e+05
19    9.905118e+05
Name: valor_ajustado, dtype: float64, residuals=0     -80605.316932
1    -109066.187252
2      44920.582324
3    -156517.638495
4     217633.882486
5    -196328.998767
6      15764.071662
7     113764.766089
8     949216.433071
9    -135174.566051
10    159278.618592
11    -81625.399244
12   -170688.418554
13   -255645.199995
14   -121844.234789
15    -44290.217662
16     69541.080225
17   -2

### Nota ao professor

No script-base da Aula 4, a validação parte de um modelo OLS construído com `preco` como variável dependente e com as variáveis explicativas preferidas disponíveis na base. [cite:68]
Isso é importante porque mostra que o diagnóstico depende diretamente da especificação do modelo que foi escolhido. [cite:68]

## Etapa 3 — Construir o relatório de diagnósticos NBR

### Mensagem-chave

Agora entramos no coração da aula: transformar o ajuste da regressão em um relatório de validação.
Essa passagem é essencial para formar o raciocínio técnico do aluno.

In [5]:
diagnostics_report, coefficients_report, summary = build_nbr_diagnostics_report(
    artifacts,
    minimum_adjusted_r_squared=0.70,
    alpha_f=0.05,
    alpha_t=0.10,
    alpha_shapiro=0.05,
    dw_lower_bound=1.5,
    dw_upper_bound=2.5,
)

## Etapa 4 — Ler o relatório consolidado de diagnósticos

### Intenção pedagógica

O professor deve mostrar que o relatório não existe para decorar indicadores.
Ele existe para organizar uma decisão técnica sobre adequação do modelo.

In [6]:
format_diagnostics_for_display(diagnostics_report)

,pressuposto,metrica_teste,valor_obtido,criterio_aceitacao,status
0,Poder Explicativo,R2 Ajustado,0.502044,R2 Ajustado >= 0.70,REPROVADO
1,Significancia Global,Teste F p-valor,0.002526,p-valor F < 0.05,APROVADO
2,Normalidade dos Residuos,Shapiro-Wilk p-valor,0.000122,p-valor W > 0.05,NAO_NORMAL
3,Independencia de Residuos,Durbin-Watson,2.176949,Entre 1.5 e 2.5,APROVADO


### Perguntas para condução

- quais itens aparecem como aprovados, normais ou críticos?
- existe coerência entre as leituras dos diferentes testes?
- o relatório sugere confiança plena, cautela ou reprovação técnica?

## Etapa 5 — Ler a significância individual dos coeficientes

### Ponto conceitual

A significância global do modelo não elimina a necessidade de olhar coeficiente por coeficiente.
Uma boa condução aqui mostra que o professor lê o modelo em duas camadas: o conjunto e cada termo explicativo.

In [7]:
format_coefficients_for_display(coefficients_report)

,variavel_explicativa,coeficiente,erro_padrao,estatistica_t,p_valor_t,significativo_10pct
0,const,-549422.227260,849629.975338,-0.646661,0.517852,NAO
1,areaprivativa,16399.584411,11491.457537,1.427111,0.153548,NAO
2,vagas,17567.447399,260162.988375,0.067525,0.946164,NAO
3,distanciacentrokm,-40893.354601,97769.177268,-0.418264,0.675754,NAO


### Fala sugerida

“O modelo pode ser relevante como conjunto e, ainda assim, conter variáveis individualmente frágeis. Por isso, a leitura do teste F não substitui a leitura do teste t.”

## Etapa 6 — Ler o resumo executivo da aprovação técnica

### Objetivo docente

Esta etapa ajuda o aluno a perceber como uma bateria de testes se converte em uma síntese técnica comunicável.

In [8]:
summary

{'status_geral_modelo': 'REPROVADO',
 'itens_aprovados_ou_normais': 2,
 'itens_avaliados': 4,
 'proporcao_aprovacao': 0.5,
 'teste_normalidade_observacao': 'Teste executado com scipy.stats.shapiro.',
 'durbin_watson_valor': 2.1769486516925984,
 'durbin_watson_faixa': '1.5 a 2.5',
 'coeficientes_significativos_10pct': 0,
 'coeficientes_totais': 4}

In [9]:
print(f"Status geral do modelo: {summary['status_geral_modelo']}")
print(
    'Itens aprovados ou normais: '
    f"{summary['itens_aprovados_ou_normais']} de {summary['itens_avaliados']}"
)
print(f"Proporção de aprovação: {summary['proporcao_aprovacao']:.2%}")
print(
    'Coeficientes significativos a 10%: '
    f"{summary['coeficientes_significativos_10pct']} de {summary['coeficientes_totais']}"
)
print('Observação sobre normalidade:', summary['teste_normalidade_observacao'])
print('Faixa usada para Durbin-Watson:', summary['durbin_watson_faixa'])
print(f"Valor observado de Durbin-Watson: {summary['durbin_watson_valor']:.6f}")

Status geral do modelo: REPROVADO
Itens aprovados ou normais: 2 de 4
Proporção de aprovação: 50.00%
Coeficientes significativos a 10%: 0 de 4
Observação sobre normalidade: Teste executado com scipy.stats.shapiro.
Faixa usada para Durbin-Watson: 1.5 a 2.5
Valor observado de Durbin-Watson: 2.176949


### Nota ao professor

Aqui vale ensinar linguagem técnica de encerramento, por exemplo:

- “o modelo apresentou aderência satisfatória aos critérios avaliados”; 
- “o modelo demanda cautela em razão de fragilidade na normalidade”; 
- “a independência residual ficou em faixa aceitável”; 
- “há necessidade de revisão de especificação”.

## Etapa 7 — Normalidade dos resíduos

### O que enfatizar

Normalidade é importante, mas não deve ser tratada de forma supersticiosa.
O professor deve lembrar que o teste produz um indício estatístico, que precisa ser lido dentro do contexto amostral e do objetivo do modelo.

### Perguntas para a turma

- o que significa dizer que os resíduos são compatíveis ou não com normalidade?
- esse resultado, isoladamente, reprova todo o modelo?
- qual seria o impacto disso na confiança inferencial?

## Etapa 8 — Independência dos resíduos e Durbin-Watson

### Mensagem pedagógica

A independência residual ajuda a verificar se os erros não estão organizados com padrão serial relevante.
Na aula, isso pode ser apresentado como uma checagem de comportamento residual compatível com um ajuste tecnicamente defensável.

### Fala sugerida

“O Durbin-Watson não é um número para decorar; ele é uma pista sobre dependência serial dos resíduos. O importante é entender se o valor observado cai ou não em uma faixa considerada aceitável.”

## Etapa 9 — Ler aprovação global com cautela

### Objetivo didático

O professor deve evitar dois extremos:

- transformar a aprovação em selo absoluto de verdade;
- transformar qualquer reprovação parcial em inutilidade total do modelo.

A boa leitura é graduada, técnica e argumentativa.

## Erros conceituais comuns

- achar que o modelo está validado apenas porque foi ajustado;
- confundir significância global com significância individual;
- tratar normalidade como tudo-ou-nada sem contexto;
- ignorar independência dos resíduos;
- comunicar resultado técnico sem explicitar critérios usados.

## Exercício supervisionado

Peça à turma que redija, com base nas saídas do notebook:

- uma frase sobre o status geral do modelo;
- uma frase sobre a significância individual dos coeficientes;
- uma frase sobre a leitura de normalidade;
- uma frase sobre a leitura do Durbin-Watson;
- uma conclusão curta sobre aderência técnica do modelo.

In [10]:
# Espaço livre para exploração guiada.
# Professor: use esta célula para pedir leituras comparativas,
# redação técnica ou discussão de aprovação parcial.

format_diagnostics_for_display(diagnostics_report)

,pressuposto,metrica_teste,valor_obtido,criterio_aceitacao,status
0,Poder Explicativo,R2 Ajustado,0.502044,R2 Ajustado >= 0.70,REPROVADO
1,Significancia Global,Teste F p-valor,0.002526,p-valor F < 0.05,APROVADO
2,Normalidade dos Residuos,Shapiro-Wilk p-valor,0.000122,p-valor W > 0.05,NAO_NORMAL
3,Independencia de Residuos,Durbin-Watson,2.176949,Entre 1.5 e 2.5,APROVADO


## Fechamento do ciclo

Sugestão de fala final:

“Com esta aula, fechamos o ciclo completo: coletamos a base, colocamos os dados em escala comparável, saneamos a amostra, ajustamos a regressão e validamos o modelo. Esse encadeamento é o que transforma cálculo em procedimento técnico defensável.”